In [ ]:
import torch

print("PyTorch version:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))


PyTorch version: 2.11.0+cu128
GPU available: True
GPU: Tesla T4


In [ ]:
!pip install -q transformers datasets peft accelerate bitsandbytes trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 64.1 MB/s eta 0:00:00


In [ ]:
import transformers
import datasets
import peft
import trl

print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)

Transformers: 5.16.1
Datasets: 4.8.5
PEFT: 0.20.0
TRL: 1.13.0


In [ ]:
from datasets import load_dataset

dataset = load_dataset("databricks/databricks-dolly-15k")

print(dataset)


README.md:   0%|          | 0.00/8.20k [00:00<?, ?B/s]

databricks-dolly-15k.jsonl: reconstructing file:   0%|          |  0.00B / 13.1MB            

databricks-dolly-15k.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'context', 'response', 'category'],
        num_rows: 15011
    })
})


In [ ]:
print(dataset["train"][0])

{'instruction': 'When did Virgin Australia start operating?', 'context': "Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.", 'response': 'Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.', 'category': 'closed_qa'}


In [ ]:
print(dataset["train"].column_names)
print("Total examples:", len(dataset["train"]))

['instruction', 'context', 'response', 'category']
Total examples: 15011


In [ ]:
example = dataset["train"][0]

print("Instruction:", example["instruction"])
print("Context:", example["context"])
print("Response:", example["response"])


Instruction: When did Virgin Australia start operating?
Context: Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.
Response: Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.


In [ ]:
def format_example(example):
    instruction = example["instruction"]
    context = example["context"]
    response = example["response"]

    if context:
        text = f"""### Instruction:
{instruction}

### Context:
{context}

### Response:
{response}"""
    else:
        text = f"""### Instruction:
{instruction}

### Response:
{response}"""

    return {"text": text}


In [ ]:
formatted = format_example(dataset["train"][0])

print(formatted["text"])


### Instruction:
When did Virgin Australia start operating?

### Context:
Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.

### Response:
Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.


In [ ]:
formatted_dataset = dataset.map(format_example)


Map:   0%|          | 0/15011 [00:00<?, ? examples/s]

In [ ]:
print(formatted_dataset)


DatasetDict({
    train: Dataset({
        features: ['instruction', 'context', 'response', 'category', 'text'],
        num_rows: 15011
    })
})


In [ ]:
print(formatted_dataset["train"][0]["text"])


### Instruction:
When did Virgin Australia start operating?

### Context:
Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.

### Response:
Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.


In [ ]:
split_dataset = formatted_dataset["train"].train_test_split(
    test_size=0.1,
    seed=42
)

In [ ]:
print(split_dataset)


DatasetDict({
    train: Dataset({
        features: ['instruction', 'context', 'response', 'category', 'text'],
        num_rows: 13509
    })
    test: Dataset({
        features: ['instruction', 'context', 'response', 'category', 'text'],
        num_rows: 1502
    })
})


In [ ]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"


In [ ]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded!")


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer loaded!


In [ ]:
print(tokenizer)


Qwen2Tokenizer(name_or_path='Qwen/Qwen2.5-1.5B-Instruct', vocab_size=151643, model_max_length=131072, padding_side='right', truncation_side='right', special_tokens={'eos_token': '<|im_end|>', 'pad_token': '<|endoftext|>'}, added_tokens_decoder={
	151643: AddedToken("<|endoftext|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151644: AddedToken("<|im_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151645: AddedToken("<|im_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151646: AddedToken("<|object_ref_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151647: AddedToken("<|object_ref_end|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151648: AddedToken("<|box_start|>", rstrip=False, lstrip=False, single_word=False, normalized=False, special=True),
	151649: AddedToken("<|box_end|>", rstrip=Fa

In [ ]:
messages = [
    {
        "role": "user",
        "content": "What is machine learning?"
    },
    {
        "role": "assistant",
        "content": "Machine learning is a method..."
    }
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=False
)

print(text)


<|im_start|>system
You are Qwen, created by Alibaba Cloud. You are a helpful assistant.<|im_end|>
<|im_start|>user
What is machine learning?<|im_end|>
<|im_start|>assistant
Machine learning is a method...<|im_end|>



In [ ]:
import torch
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)


In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)


model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [ ]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto",
    output_hidden_states=True
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
print("Model loaded!")


Model loaded!


In [ ]:
print(torch.cuda.get_device_name(0))


Tesla T4


In [ ]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)


In [ ]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)


In [ ]:
from peft import get_peft_model

model = get_peft_model(
    model,
    lora_config
)


In [ ]:
model.print_trainable_parameters()


trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


In [ ]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        truncation=True,
        max_length=512
    )


In [ ]:
tokenized_dataset = split_dataset.map(
    tokenize_function,
    batched=True,
    remove_columns=split_dataset["train"].column_names
)


Map:   0%|          | 0/13509 [00:00<?, ? examples/s]

Map:   0%|          | 0/1502 [00:00<?, ? examples/s]

In [ ]:
print(tokenized_dataset)


DatasetDict({
    train: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 13509
    })
    test: Dataset({
        features: ['input_ids', 'attention_mask'],
        num_rows: 1502
    })
})


In [ ]:
print(tokenized_dataset["train"][0].keys())


dict_keys(['input_ids', 'attention_mask'])


In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)


In [ ]:
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print("Train:", len(train_dataset))
print("Validation:", len(eval_dataset))
print(train_dataset[0]["text"])


Train: 13509
Validation: 1502
### Instruction:
Given the reference text below the eruption of Mount Vesuvius, how high was the mountain after the disaster?

### Context:
In December 1631, Mount Vesuvius in Italy erupted. The eruption began on 16 December 1631 and culminated the day after. The Volcanic Explosivity Index was VEI-5, and it was a Plinian eruption that buried many villages under the resulting lava flows. It is estimated that between 4,000 people were killed by the eruption, making it the highest death toll for a volcanic disaster in the Mediterranean in the last 1800 years.[citation needed] The 1631 eruption was considered to be of minor proportions regarding its eruptive magnitude and erupted volumes compared to the AD 79 eruption, but the damage was not.[citation needed] By the 1631 eruption, the summit of Mount Vesuvius had been reduced by 450m, making its total height lower than that of Mount Somma.

### Response:
Mount Vesuvius had a reduced summit by 450 meters.


In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./qwen-lora",

    num_train_epochs=1,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    gradient_accumulation_steps=4,

    learning_rate=2e-4,

    logging_steps=10,

    save_steps=100,
    eval_steps=100,

    fp16=True,

    report_to="none"
)


In [ ]:
from trl import SFTTrainer


In [ ]:
model = get_peft_model(model, lora_config)


/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:72: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:78: UserWarning: The PEFT config's `base_model_name_or_path` was renamed from 'Qwen/Qwen2.5-1.5B-Instruct' to 'None'. Please ensure that the correct base model is loaded when loading this checkpoint.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:305: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [1]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

ModuleNotFoundError: No module named 'trl'

In [2]:
!pip install -U -q trl transformers datasets peft accelerate bitsandbytes

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.0/925.0 kB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 41.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 11.7 MB/s eta 0:00:00


In [ ]:
peft_config=lora_config


In [3]:
import torch
import transformers
import datasets
import peft
import trl

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
Transformers: 5.17.0
Datasets: 5.0.1
PEFT: 0.21.0
TRL: 1.14.0
CUDA: True
GPU: Tesla T4


In [6]:
model = get_peft_model(model, lora_config)

NameError: name 'get_peft_model' is not defined

In [7]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

NameError: name 'model' is not defined

In [ ]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)


Adding EOS to train dataset:   0%|          | 0/13509 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/13509 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/13509 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/13509 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/13509 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/1502 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1502 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/1502 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1502 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/1502 [00:00<?, ? examples/s]

In [8]:
from peft import get_peft_model, LoraConfig

In [ ]:
model.print_trainable_parameters()


trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


In [9]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import (
    LoraConfig,
    get_peft_model,
    prepare_model_for_kbit_training
)

In [11]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

In [12]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

In [13]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

In [14]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

In [15]:
model = prepare_model_for_kbit_training(model)

In [16]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

In [17]:
model = get_peft_model(model, lora_config)

model.print_trainable_parameters()

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


In [18]:
from datasets import load_dataset

dataset = load_dataset("databricks/databricks-dolly-15k")

README.md:   0%|          | 0.00/8.20k [00:00<?, ?B/s]

databricks-dolly-15k.jsonl: reconstructing file:   0%|          |  0.00B / 13.1MB            

databricks-dolly-15k.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

In [19]:
model = get_peft_model(model, lora_config)

/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:148: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:154: UserWarning: The PEFT config's `base_model_name_or_path` was renamed from 'Qwen/Qwen2.5-1.5B-Instruct' to 'None'. Please ensure that the correct base model is loaded when loading this checkpoint.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [20]:
peft_config=lora_config

In [23]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./qwen-lora",

    num_train_epochs=1,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    gradient_accumulation_steps=4,

    learning_rate=2e-4,

    logging_steps=10,

    save_steps=100,
    eval_steps=100,

    fp16=True,

    report_to="none"
)

print("Training arguments created successfully!")

Training arguments created successfully!


In [28]:
from datasets import load_dataset

dataset = load_dataset("databricks/databricks-dolly-15k")

print(dataset)
print("Total examples:", len(dataset["train"]))

DatasetDict({
    train: Dataset({
        features: ['instruction', 'context', 'response', 'category'],
        num_rows: 15011
    })
})
Total examples: 15011


In [29]:
def format_example(example):
    instruction = example["instruction"]
    context = example["context"]
    response = example["response"]

    if context:
        text = f"""### Instruction:
{instruction}

### Context:
{context}

### Response:
{response}"""
    else:
        text = f"""### Instruction:
{instruction}

### Response:
{response}"""

    return {"text": text}

In [30]:
formatted_dataset = dataset.map(format_example)

print(formatted_dataset)
print(formatted_dataset["train"][0]["text"])

Map:   0%|          | 0/15011 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'context', 'response', 'category', 'text'],
        num_rows: 15011
    })
})
### Instruction:
When did Virgin Australia start operating?

### Context:
Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.

### Response:
Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.


In [31]:
split_dataset = formatted_dataset["train"].train_test_split(
    test_size=0.1,
    seed=42
)

print(split_dataset)

DatasetDict({
    train: Dataset({
        features: ['instruction', 'context', 'response', 'category', 'text'],
        num_rows: 13509
    })
    test: Dataset({
        features: ['instruction', 'context', 'response', 'category', 'text'],
        num_rows: 1502
    })
})


In [32]:
train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print("Train:", len(train_dataset))
print("Validation:", len(eval_dataset))

Train: 13509
Validation: 1502


In [33]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

Adding EOS to train dataset:   0%|          | 0/13509 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/13509 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/13509 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/13509 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/13509 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/1502 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1502 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/1502 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1502 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/1502 [00:00<?, ? examples/s]

In [35]:
!pip uninstall -y trl
!pip install -q "trl==0.12.0"

Found existing installation: trl 1.14.0
Uninstalling trl-1.14.0:
  Successfully uninstalled trl-1.14.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 310.2/310.2 kB 7.2 MB/s eta 0:00:00


In [36]:
import trl
print(trl.__version__)

1.14.0


In [37]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="./qwen-lora",

    num_train_epochs=1,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    gradient_accumulation_steps=4,

    learning_rate=2e-4,

    logging_steps=10,

    save_steps=100,
    eval_steps=100,

    fp16=True,

    report_to="none",

    loss_type="nll",
)

In [38]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
)

In [40]:
print(type(training_args))

if hasattr(training_args, "loss_type"):
    print("loss_type:", training_args.loss_type)

print(type(model))

<class 'trl.trainer.sft_config.SFTConfig'>
loss_type: nll
<class 'peft.peft_model.PeftModelForCausalLM'>


In [41]:
model = get_peft_model(model, lora_config)

/usr/local/lib/python3.13/dist-packages/peft/mapping_func.py:148: UserWarning: You are trying to modify a model with PEFT for a second time. If you want to reload the model with a different config, make sure to call `.unload()` before.
  warnings.warn(
/usr/local/lib/python3.13/dist-packages/peft/tuners/tuners_utils.py:331: UserWarning: Already found a `peft_config` attribute in the model. This will lead to having multiple adapters in the model. Make sure to know what you are doing!
  warnings.warn(


In [42]:
from transformers import AutoModelForCausalLM, BitsAndBytesConfig
import torch

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [43]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

In [44]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

In [45]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="./qwen-lora",

    num_train_epochs=1,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    gradient_accumulation_steps=4,

    learning_rate=2e-4,

    logging_steps=10,

    save_steps=100,
    eval_steps=100,

    fp16=True,

    report_to="none",

    loss_type="nll",

    max_length=512,

    eos_token="<|im_end|>",
)

In [46]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=eval_dataset,

    processing_class=tokenizer,

    peft_config=lora_config,
)

Tokenizing train dataset:   0%|          | 0/13509 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/13509 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/13509 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/13509 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1502 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/1502 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1502 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/1502 [00:00<?, ? examples/s]

In [47]:
print("Loss type:", trainer.args.loss_type)
print("Model type:", type(trainer.model))

Loss type: nll
Model type: <class 'peft.peft_model.PeftModelForCausalLM'>


In [48]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


NotImplementedError: "_amp_foreach_non_finite_check_and_unscale_cuda" not implemented for 'BFloat16'

In [49]:
import torch

print(torch.__version__)
print(torch.cuda.get_device_name(0))
print(torch.cuda.is_bf16_supported())

2.11.0+cu128
Tesla T4
True


In [50]:
import torch
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

In [51]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    quantization_config=bnb_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [52]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

In [59]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

In [60]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="./qwen-lora",

    num_train_epochs=1,

    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,

    gradient_accumulation_steps=4,

    learning_rate=2e-4,

    logging_steps=10,
    save_steps=100,
    eval_steps=100,

    fp16=True,
    bf16=False,

    report_to="none",

    loss_type="nll",

    max_length=512,
)

In [61]:
print("FP16:", training_args.fp16)
print("BF16:", training_args.bf16)
print("Loss:", training_args.loss_type)

FP16: True
BF16: False
Loss: nll


In [62]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

In [64]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
)

In [65]:
print(type(model))
print("Loss:", training_args.loss_type)

<class 'peft.peft_model.PeftModelForCausalLM'>
Loss: nll


In [67]:
import torch
from transformers import BitsAndBytesConfig

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

In [68]:
from transformers import AutoModelForCausalLM

model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    quantization_config=bnb_config,
    device_map="auto"
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [69]:
from peft import prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

In [70]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir="./qwen-lora",

    num_train_epochs=1,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=8,

    learning_rate=2e-4,

    logging_steps=10,
    save_steps=100,
    eval_steps=100,

    fp16=False,
    bf16=False,

    gradient_checkpointing=True,

    report_to="none",

    loss_type="nll",
    max_length=512,
)

In [71]:
from peft import LoraConfig

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

In [72]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

In [73]:
print("FP16:", trainer.args.fp16)
print("BF16:", trainer.args.bf16)
print("Loss:", trainer.args.loss_type)

FP16: False
BF16: False
Loss: nll


In [74]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss
10,2.372431
20,2.209649
30,1.969062
40,2.105784
50,1.979652
60,2.041733
70,1.997438
80,1.856519
90,1.955635
100,1.840612


TrainOutput(global_step=1689, training_loss=1.919156104449763, metrics={'train_runtime': 7078.1682, 'train_samples_per_second': 1.909, 'train_steps_per_second': 0.239, 'total_flos': 1.713268084451328e+16, 'train_loss': 1.919156104449763, 'entropy': 1.918193157168402, 'mean_token_accuracy': 0.5721866523010143, 'num_tokens': 2171940.0, 'epoch': 1.0})

In [1]:
trainer.save_model("./qwen-dolly-lora")
tokenizer.save_pretrained("./qwen-dolly-lora")

NameError: name 'trainer' is not defined

In [2]:
print("trainer" in globals())

False


In [3]:
import os

print(os.listdir("./qwen-lora"))

FileNotFoundError: [Errno 2] No such file or directory: './qwen-lora'

In [4]:
save_steps=100

In [5]:
import os

print(os.listdir("./qwen-lora"))

FileNotFoundError: [Errno 2] No such file or directory: './qwen-lora'

In [6]:
a

NameError: name 'a' is not defined

In [7]:
save_steps=100

In [8]:
import os

print("Current directory:", os.getcwd())
print("Files:")
print(os.listdir("/content"))

Current directory: /content
Files:
['.config', 'sample_data']


In [9]:
import os

for root, dirs, files in os.walk("/content"):
    for d in dirs:
        if "checkpoint" in d.lower() or "qwen" in d.lower():
            print(os.path.join(root, d))

In [10]:
import os

for root, dirs, files in os.walk("/content"):
    for d in dirs:
        if "checkpoint" in d.lower() or "qwen" in d.lower():
            print(os.path.join(root, d))

In [11]:
import os

print(os.path.exists("/content/drive"))

False


In [12]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [13]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for d in dirs:
        if "checkpoint" in d.lower() or "qwen" in d.lower() or "lora" in d.lower():
            print(os.path.join(root, d))

In [14]:
import os

for root, dirs, files in os.walk("/content/drive/MyDrive"):
    for d in dirs:
        if "checkpoint" in d.lower() or "qwen" in d.lower() or "lora" in d.lower():
            print(os.path.join(root, d))

In [15]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
output_dir = "/content/drive/MyDrive/qwen-dolly-lora"

In [18]:
from trl import SFTConfig

ModuleNotFoundError: No module named 'trl'

In [19]:
!pip install -q -U \
  "trl==1.14.0" \
  "transformers" \
  "datasets" \
  "peft" \
  "accelerate" \
  "bitsandbytes"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.0/925.0 kB 24.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 56.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 17.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 30.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 34.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 19.1 MB/s eta 0:00:00


In [20]:
import torch
import trl
import transformers
import peft

print("PyTorch:", torch.__version__)
print("TRL:", trl.__version__)
print("Transformers:", transformers.__version__)
print("PEFT:", peft.__version__)
print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
TRL: 1.14.0
Transformers: 5.17.0
PEFT: 0.21.0
GPU: Tesla T4


In [21]:
from trl import SFTConfig, SFTTrainer

In [22]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
output_dir = "/content/drive/MyDrive/qwen-dolly-lora"

In [24]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [25]:
output_dir = "/content/drive/MyDrive/qwen-dolly-lora"

import os
os.makedirs(output_dir, exist_ok=True)

print(output_dir)

/content/drive/MyDrive/qwen-dolly-lora


In [26]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import (
    LoraConfig,
    prepare_model_for_kbit_training
)

from trl import SFTConfig, SFTTrainer

print("All imports successful!")

All imports successful!


In [27]:
print("PyTorch:", torch.__version__)
print("GPU:", torch.cuda.get_device_name(0))
print("CUDA:", torch.cuda.is_available())

PyTorch: 2.11.0+cu128
GPU: Tesla T4
CUDA: True


In [28]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("Tokenizer loaded!")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

Tokenizer loaded!


In [29]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

In [30]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("Model loaded!")

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

Model loaded!


In [31]:
model = prepare_model_for_kbit_training(model)

In [32]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj"
    ]
)

In [33]:
from datasets import load_dataset

dataset = load_dataset("databricks/databricks-dolly-15k")

print(dataset)
print("Total examples:", len(dataset["train"]))

README.md:   0%|          | 0.00/8.20k [00:00<?, ?B/s]

databricks-dolly-15k.jsonl: reconstructing file:   0%|          |  0.00B / 13.1MB            

databricks-dolly-15k.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'context', 'response', 'category'],
        num_rows: 15011
    })
})
Total examples: 15011


In [34]:
def format_example(example):
    instruction = example["instruction"]
    context = example["context"]
    response = example["response"]

    if context:
        text = f"""### Instruction:
{instruction}

### Context:
{context}

### Response:
{response}"""
    else:
        text = f"""### Instruction:
{instruction}

### Response:
{response}"""

    return {"text": text}

In [35]:
formatted_dataset = dataset.map(format_example)

print(formatted_dataset)
print(formatted_dataset["train"][0]["text"])

Map:   0%|          | 0/15011 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['instruction', 'context', 'response', 'category', 'text'],
        num_rows: 15011
    })
})
### Instruction:
When did Virgin Australia start operating?

### Context:
Virgin Australia, the trading name of Virgin Australia Airlines Pty Ltd, is an Australian-based airline. It is the largest airline by fleet size to use the Virgin brand. It commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route. It suddenly found itself as a major airline in Australia's domestic market after the collapse of Ansett Australia in September 2001. The airline has since grown to directly serve 32 cities in Australia, from hubs in Brisbane, Melbourne and Sydney.

### Response:
Virgin Australia commenced services on 31 August 2000 as Virgin Blue, with two aircraft on a single route.


In [36]:
split_dataset = formatted_dataset["train"].train_test_split(
    test_size=0.1,
    seed=42
)

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print("Train:", len(train_dataset))
print("Validation:", len(eval_dataset))

Train: 13509
Validation: 1502


In [37]:
from trl import SFTConfig

training_args = SFTConfig(
    output_dir=output_dir,

    num_train_epochs=1,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=8,

    learning_rate=2e-4,

    logging_steps=10,

    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,

    fp16=False,
    bf16=False,

    report_to="none",

    loss_type="nll",

    max_length=512,

    eos_token="<|im_end|>",
)

In [38]:
save_steps=100

In [39]:
from trl import SFTTrainer

trainer = SFTTrainer(
    model=model,
    args=training_args,

    train_dataset=train_dataset,
    eval_dataset=eval_dataset,

    processing_class=tokenizer,

    peft_config=lora_config,
)

Adding EOS to train dataset:   0%|          | 0/13509 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/13509 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/13509 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/13509 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/13509 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/1502 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/1502 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/1502 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1502 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/1502 [00:00<?, ? examples/s]

In [40]:
print("Output:", training_args.output_dir)
print("Loss:", training_args.loss_type)
print("FP16:", training_args.fp16)
print("BF16:", training_args.bf16)

Output: /content/drive/MyDrive/qwen-dolly-lora
Loss: nll
FP16: False
BF16: False


In [41]:
print(type(trainer.model))

<class 'peft.peft_model.PeftModelForCausalLM'>


In [42]:
print(trainer.args.output_dir)

/content/drive/MyDrive/qwen-dolly-lora


In [43]:
print("Trainer model type:", type(trainer.model))
print("Output directory:", trainer.args.output_dir)
print("Loss type:", trainer.args.loss_type)
print("FP16:", trainer.args.fp16)
print("BF16:", trainer.args.bf16)

Trainer model type: <class 'peft.peft_model.PeftModelForCausalLM'>
Output directory: /content/drive/MyDrive/qwen-dolly-lora
Loss type: nll
FP16: False
BF16: False


In [4]:
trainer.train()

NameError: name 'trainer' is not defined

In [5]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [6]:
import os

output_dir = "/content/drive/MyDrive/qwen-dolly-lora"

print("Exists:", os.path.exists(output_dir))

if os.path.exists(output_dir):
    print("Files:")
    for x in sorted(os.listdir(output_dir)):
        print(x)

Exists: True
Files:
README.md
checkpoint-300
checkpoint-400


In [7]:
import os

output_dir = "/content/drive/MyDrive/qwen-dolly-lora"

checkpoints = [
    x for x in os.listdir(output_dir)
    if x.startswith("checkpoint-")
]

checkpoints = sorted(checkpoints, key=lambda x: int(x.split("-")[1]))

print("Checkpoints:", checkpoints)
print("Latest:", checkpoints[-1])

Checkpoints: ['checkpoint-300', 'checkpoint-400']
Latest: checkpoint-400


In [8]:
latest_checkpoint = os.path.join(output_dir, checkpoints[-1])

print(latest_checkpoint)

/content/drive/MyDrive/qwen-dolly-lora/checkpoint-400


In [10]:
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import (
    LoraConfig,
    prepare_model_for_kbit_training
)

from trl import SFTConfig, SFTTrainer

print("✅ All imports successful!")

ModuleNotFoundError: No module named 'trl'

In [11]:
!pip install -q "trl==1.14.0" "transformers" "datasets" "peft" "accelerate" "bitsandbytes"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 925.0/925.0 kB 25.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 27.1 MB/s eta 0:00:00


In [12]:
import trl

print("TRL version:", trl.__version__)

TRL version: 1.14.0


In [13]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [14]:
import os
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)

from peft import (
    LoraConfig,
    prepare_model_for_kbit_training
)

from trl import SFTConfig, SFTTrainer

print("✅ Imports successful!")
print("PyTorch:", torch.__version__)
print("TRL:", __import__("trl").__version__)
print("GPU:", torch.cuda.get_device_name(0))

✅ Imports successful!
PyTorch: 2.11.0+cu128
TRL: 1.14.0
GPU: Tesla T4


In [15]:
output_dir = "/content/drive/MyDrive/qwen-dolly-lora"
latest_checkpoint = "/content/drive/MyDrive/qwen-dolly-lora/checkpoint-400"

print("Checkpoint exists:", os.path.exists(latest_checkpoint))
print("Checkpoint:", latest_checkpoint)

Checkpoint exists: True
Checkpoint: /content/drive/MyDrive/qwen-dolly-lora/checkpoint-400


In [16]:
model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

print("✅ Tokenizer loaded")

config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

✅ Tokenizer loaded


In [17]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

print("✅ 4-bit config ready")

✅ 4-bit config ready


In [18]:
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

print("✅ Base model loaded")

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Base model loaded


In [19]:
 model = prepare_model_for_kbit_training(model)

print("✅ Model prepared for k-bit training")

✅ Model prepared for k-bit training


In [20]:
trainer.train(resume_from_checkpoint=latest_checkpoint)

NameError: name 'trainer' is not defined

In [21]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

print("✅ LoRA config ready")

✅ LoRA config ready


In [22]:
from datasets import load_dataset

dataset = load_dataset("databricks/databricks-dolly-15k")

print("Total examples:", len(dataset["train"]))

README.md:   0%|          | 0.00/8.20k [00:00<?, ?B/s]

databricks-dolly-15k.jsonl: reconstructing file:   0%|          |  0.00B / 13.1MB            

databricks-dolly-15k.jsonl: downloading bytes:           |  0.00B            

Generating train split:   0%|          | 0/15011 [00:00<?, ? examples/s]

Total examples: 15011


In [23]:
def format_example(example):
    instruction = example["instruction"]
    context = example["context"]
    response = example["response"]

    if context:
        text = f"""### Instruction:
{instruction}

### Context:
{context}

### Response:
{response}"""
    else:
        text = f"""### Instruction:
{instruction}

### Response:
{response}"""

    return {"text": text}

formatted_dataset = dataset.map(format_example)

print("✅ Dataset formatted")

Map:   0%|          | 0/15011 [00:00<?, ? examples/s]

✅ Dataset formatted


In [24]:
split_dataset = formatted_dataset["train"].train_test_split(
    test_size=0.1,
    seed=42
)

train_dataset = split_dataset["train"]
eval_dataset = split_dataset["test"]

print("Train:", len(train_dataset))
print("Validation:", len(eval_dataset))

Train: 13509
Validation: 1502


In [25]:
training_args = SFTConfig(
    output_dir=output_dir,

    num_train_epochs=1,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,

    gradient_accumulation_steps=8,

    learning_rate=2e-4,

    logging_steps=10,

    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,

    fp16=False,
    bf16=False,

    report_to="none",

    loss_type="nll",

    max_length=512,

    eos_token="<|im_end|>",
)

print("✅ Training config ready")

✅ Training config ready


In [26]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    processing_class=tokenizer,
    peft_config=lora_config,
)

print("✅ Trainer created")

Adding EOS to train dataset:   0%|          | 0/13509 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/13509 [00:00<?, ? examples/s]

Building labels for train dataset:   0%|          | 0/13509 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/13509 [00:00<?, ? examples/s]

Dropping fully masked examples from train dataset:   0%|          | 0/13509 [00:00<?, ? examples/s]

...

Tokenizing eval dataset:   0%|          | 0/1502 [00:00<?, ? examples/s]

Building labels for eval dataset:   0%|          | 0/1502 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/1502 [00:00<?, ? examples/s]

Dropping fully masked examples from eval dataset:   0%|          | 0/1502 [00:00<?, ? examples/s]

✅ Trainer created


In [27]:
print("Output:", trainer.args.output_dir)
print("Loss:", trainer.args.loss_type)
print("FP16:", trainer.args.fp16)
print("BF16:", trainer.args.bf16)
print("Checkpoint:", latest_checkpoint)

Output: /content/drive/MyDrive/qwen-dolly-lora
Loss: nll
FP16: False
BF16: False
Checkpoint: /content/drive/MyDrive/qwen-dolly-lora/checkpoint-400


In [28]:
trainer.train(
    resume_from_checkpoint="/content/drive/MyDrive/qwen-dolly-lora/checkpoint-400"
)

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.


Step,Training Loss
410,1.851029
420,1.957450
430,1.784338
440,1.858942
450,2.000644
460,1.994824
470,1.958307
480,1.878199
490,1.968425
500,1.939676


TrainOutput(global_step=1689, training_loss=1.4533101731746982, metrics={'train_runtime': 5480.6596, 'train_samples_per_second': 2.465, 'train_steps_per_second': 0.308, 'total_flos': 1.713268084451328e+16, 'train_loss': 1.4533101731746982, 'entropy': 1.9168759152509165, 'mean_token_accuracy': 0.5741653014784274, 'num_tokens': 1658066.0, 'epoch': 1.0})

In [29]:
print("Training completed!")
print("Final output:", output_dir)

Training completed!
Final output: /content/drive/MyDrive/qwen-dolly-lora


In [30]:
import os

print("Output directory contents:")
for item in sorted(os.listdir(output_dir)):
    print(item)

Output directory contents:
README.md
checkpoint-1600
checkpoint-1689


In [31]:
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print("✅ Final LoRA adapter saved!")

✅ Final LoRA adapter saved!


In [32]:
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print("✅ Final LoRA adapter saved!")

✅ Final LoRA adapter saved!


In [33]:
import os

print("Files in final output directory:")
for item in sorted(os.listdir(output_dir)):
    print(item)

Files in final output directory:
README.md
adapter_config.json
adapter_model.safetensors
chat_template.jinja
checkpoint-1600
checkpoint-1689
tokenizer.json
tokenizer_config.json
training_args.bin


In [34]:
trainer.model.print_trainable_parameters()

trainable params: 4,358,144 || all params: 1,548,072,448 || trainable%: 0.2815


In [35]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

base_model_name = "Qwen/Qwen2.5-1.5B-Instruct"
adapter_path = "/content/drive/MyDrive/qwen-dolly-lora"

tokenizer = AutoTokenizer.from_pretrained(adapter_path)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

model = PeftModel.from_pretrained(
    base_model,
    adapter_path
)

model.eval()

print("✅ Fine-tuned model loaded!")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✅ Fine-tuned model loaded!


In [36]:
prompt = "Explain machine learning in simple words."

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

response = tokenizer.decode(
    outputs[0],
    skip_special_tokens=True
)

print(response)

Explain machine learning in simple words. Machine learning is a branch of artificial intelligence where machines are trained to perform tasks by analyzing and understanding patterns in data.
In other words, it's the ability of computer systems to learn from experience without being explicitly programmed. 
Machine learning algorithms can be used for various tasks such as prediction, classification, clustering, anomaly detection, regression, optimization, etc. They use statistical models or mathematical functions to make predictions based on past data inputs and outputs. 
For example, you might train a model using historical stock prices to predict future stock price movements. Or you could build an algorithm that classifies emails into spam or not spam categories. 
The more data you have, the better your machine learning model will become at making accurate predictions. This is why machine learning


In [37]:
prompt = """### Instruction:
What is machine learning?

### Response:"""

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=150,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
        pad_token_id=tokenizer.eos_token_id
    )

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

### Instruction:
What is machine learning?

### Response: 
Machine learning is a type of artificial intelligence that involves creating algorithms that can learn from data and make predictions or decisions based on the patterns they discover. It enables machines to improve their performance over time without explicit programming, making it possible for them to perform tasks more efficiently than humans. Machine learning has applications in various fields such as healthcare, finance, transportation, and manufacturing, where it can help optimize processes, reduce errors, and improve decision-making.


In [38]:
test_prompts = [
    "What is machine learning?",
    "Explain overfitting in simple words.",
    "What is the difference between supervised and unsupervised learning?",
    "Why is Python popular for data science?",
    "What are the advantages of cloud computing?"
]

print("Number of test prompts:", len(test_prompts))

Number of test prompts: 5


In [39]:
def generate_response(prompt):
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

for i, prompt in enumerate(test_prompts, 1):
    print(f"\n{'='*60}")
    print(f"TEST {i}")
    print("Question:", prompt)
    print("Answer:", generate_response(prompt))


TEST 1
Question: What is machine learning?
Answer: What is machine learning? Machine Learning (ML) refers to a class of algorithms that enable computer systems to learn from data. This enables the system to make decisions and predictions based on this learned information.
Machine learning can be divided into two main categories: supervised learning and unsupervised learning.
Supervised learning involves training an algorithm using labeled examples, where each example contains both input features and corresponding output labels. The goal is to develop a model that can predict the correct label for new, unlabeled examples. Supervised learning tasks include classification and regression problems. An example of supervised learning in action would be building a spam filter to classify emails as either spam or not spam.
Unsupervised learning, on the other hand, focuses on finding patterns and relationships within unlabelled data

TEST 2
Question: Explain overfitting in simple words.
Answer:

In [2]:
def generate_response(prompt):
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

for i, prompt in enumerate(test_prompts, 1):
    print(f"\n{'='*60}")
    print(f"TEST {i}")
    print("Question:", prompt)
    print("Answer:", generate_response(prompt))

NameError: name 'test_prompts' is not defined

In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
import os
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import PeftModel

# =========================
# 1. Paths
# =========================

base_model_name = "Qwen/Qwen2.5-1.5B-Instruct"
adapter_path = "/content/drive/MyDrive/qwen-dolly-lora"

print("Adapter exists:", os.path.exists(adapter_path))

# =========================
# 2. Load tokenizer
# =========================

tokenizer = AutoTokenizer.from_pretrained(adapter_path)

# =========================
# 3. 4-bit configuration
# =========================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# =========================
# 4. Load base model
# =========================

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

# =========================
# 5. Load LoRA adapter
# =========================

model = PeftModel.from_pretrained(
    base_model,
    adapter_path
)

model.eval()

print("✅ Fine-tuned model loaded!")

# =========================
# 6. Test prompts
# =========================

test_prompts = [
    "What is machine learning?",
    "Explain overfitting in simple words.",
    "What is the difference between supervised and unsupervised learning?",
    "Why is Python popular for data science?",
    "What are the advantages of cloud computing?"
]

# =========================
# 7. Generate function
# =========================

def generate_response(prompt):
    inputs = tokenizer(
        prompt,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    return tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )

# =========================
# 8. Run tests
# =========================

for i, prompt in enumerate(test_prompts, 1):

    print("\n" + "=" * 70)
    print(f"TEST {i}")
    print("QUESTION:", prompt)
    print("-" * 70)

    answer = generate_response(prompt)

    print("ANSWER:")
    print(answer)

Adapter exists: True


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

In [5]:
!pip install -q -U "bitsandbytes>=0.46.1"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 15.6 MB/s eta 0:00:00


In [6]:
import bitsandbytes as bnb

print("bitsandbytes:", bnb.__version__)

bitsandbytes: 0.50.2


In [7]:
import os
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import PeftModel

# =========================
# Paths
# =========================

base_model_name = "Qwen/Qwen2.5-1.5B-Instruct"
adapter_path = "/content/drive/MyDrive/qwen-dolly-lora"

# =========================
# Check adapter
# =========================

print("Adapter exists:", os.path.exists(adapter_path))

# =========================
# Tokenizer
# =========================

tokenizer = AutoTokenizer.from_pretrained(
    base_model_name
)

# =========================
# 4-bit QLoRA config
# =========================

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# =========================
# Load base model
# =========================

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

# =========================
# Load trained LoRA adapter
# =========================

model = PeftModel.from_pretrained(
    base_model,
    adapter_path
)

model.eval()

print("✅ Fine-tuned Qwen model loaded!")

# =========================
# Generation function
# =========================

def generate_response(prompt):

    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=150,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Only decode newly generated tokens
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

# =========================
# Test
# =========================

questions = [
    "What is machine learning?",
    "Explain overfitting in simple words.",
    "What is the difference between supervised and unsupervised learning?"
]

for i, question in enumerate(questions, 1):

    print("\n" + "=" * 70)
    print(f"TEST {i}")
    print("QUESTION:", question)
    print("-" * 70)
    print("ANSWER:")
    print(generate_response(question))

Adapter exists: True


tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

In [8]:
import sys
import bitsandbytes as bnb
import transformers

print("Python:", sys.version)
print("bitsandbytes:", bnb.__version__)
print("bitsandbytes path:", bnb.__file__)
print("Transformers:", transformers.__version__)

try:
    from transformers.utils import is_bitsandbytes_available
    print("Transformers sees bitsandbytes:", is_bitsandbytes_available())
except Exception as e:
    print("Check error:", e)

Python: 3.13.15 (main, Aug  6 2026, 11:06:22) [GCC 13.3.0]
bitsandbytes: 0.50.2
bitsandbytes path: /usr/local/lib/python3.13/dist-packages/bitsandbytes/__init__.py
Transformers: 5.16.1
Transformers sees bitsandbytes: False


In [9]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cpu
CUDA available: False


In [10]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cpu
CUDA: False


In [11]:
import torch
import bitsandbytes as bnb

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())
print("bitsandbytes:", bnb.__version__)

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cpu
CUDA available: False
bitsandbytes: 0.50.2


In [12]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cpu
CUDA: False


In [1]:
import torch
print(torch.cuda.is_available())
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU")

False
No GPU


In [2]:
import torch

print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

CUDA: False


In [1]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4


In [3]:
from google.colab import drive
drive.mount("/content/drive")


import os
import torch

from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig
)
from peft import PeftModel

base_model_name = "Qwen/Qwen2.5-1.5B-Instruct"
adapter_path = "/content/drive/MyDrive/qwen-dolly-lora"

print("Adapter exists:", os.path.exists(adapter_path))

# Tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_model_name)

# 4-bit QLoRA config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True
)

# Base model
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

# LoRA adapter
model = PeftModel.from_pretrained(
    base_model,
    adapter_path
)

model.eval()

print("✅ Fine-tuned model loaded!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Adapter exists: True


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

ImportError: Using `bitsandbytes` 4-bit quantization requires bitsandbytes: `pip install -U bitsandbytes>=0.46.1`

In [5]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
CUDA: True
GPU: Tesla T4


In [7]:
import os
import torch

from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base_model_name = "Qwen/Qwen2.5-1.5B-Instruct"
adapter_path = "/content/drive/MyDrive/qwen-dolly-lora"

print("Adapter exists:", os.path.exists(adapter_path))

tokenizer = AutoTokenizer.from_pretrained(base_model_name)

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    dtype=torch.float16,
    device_map="auto"
)

model = PeftModel.from_pretrained(
    base_model,
    adapter_path
)

model.eval()

print("✅ Fine-tuned model loaded successfully!")

Adapter exists: True


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

In [8]:
%pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [9]:
import importlib.metadata as metadata

try:
    print("torchao version:", metadata.version("torchao"))
except metadata.PackageNotFoundError:
    print("✅ torchao successfully uninstalled")

✅ torchao successfully uninstalled


In [10]:
import torch
import importlib.metadata as metadata

print("CUDA:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

try:
    print("torchao:", metadata.version("torchao"))
except metadata.PackageNotFoundError:
    print("torchao: Not installed ✅")

CUDA: True
GPU: Tesla T4
torchao: Not installed ✅


In [11]:
from google.colab import drive
drive.mount("/content/drive")

import os
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel

base_name = "Qwen/Qwen2.5-1.5B-Instruct"
adapter_path = "/content/drive/MyDrive/qwen-dolly-lora"

# Verify adapter files
assert os.path.isfile(
    os.path.join(adapter_path, "adapter_config.json")
), "Adapter config not found!"

assert os.path.isfile(
    os.path.join(adapter_path, "adapter_model.safetensors")
), "Adapter weights not found!"

# Load tokenizer
tokenizer = AutoTokenizer.from_pretrained(base_name)

# Load base model without bitsandbytes
base_model = AutoModelForCausalLM.from_pretrained(
    base_name,
    dtype=torch.float16,
    device_map="auto"
)

# Attach trained LoRA adapter
model = PeftModel.from_pretrained(
    base_model,
    adapter_path
)

model.eval()

print("✅ Fine-tuned Qwen loaded successfully!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✅ Fine-tuned Qwen loaded successfully!


In [12]:
def generate_response(prompt, max_new_tokens=150):
    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    # Only generated response
    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

In [13]:
question = "What is machine learning?"

answer = generate_response(question)

print("Question:", question)
print("\nAnswer:\n", answer)

Question: What is machine learning?

Answer:
 Machine Learning (ML) refers to the use of algorithms and statistical models that allow computers to automatically learn from data without being explicitly programmed. ML can be used for tasks such as image recognition, speech recognition, natural language processing, and decision making.

In general, there are two types of ML: supervised learning and unsupervised learning. In supervised learning, the algorithm learns from labeled training data where the correct output or label is provided. The goal is to find a function that maps inputs to outputs based on the training data. This type of learning is often used in applications like image classification, speech recognition, and spam detection.

Unsupervised learning, on the other hand, involves finding patterns or relationships within unlabeled data. For example, clustering algorithms


In [14]:
questions = [
    "Explain overfitting in simple words.",
    "What is the difference between supervised and unsupervised learning?",
    "Why is Python popular for data science?",
    "What are the advantages of cloud computing?"
]

for i, question in enumerate(questions, 1):
    print("\n" + "=" * 70)
    print(f"TEST {i}")
    print("Question:", question)
    print("-" * 70)
    print(generate_response(question))


TEST 1
Question: Explain overfitting in simple words.
----------------------------------------------------------------------
Overfitting is when your model performs well on the training data but poorly on new, unseen data (validation set). It happens because you have too many parameters relative to the amount of available training data. Overfitting leads to poor generalization and makes it hard for your model to learn the underlying patterns of the data. To avoid overfitting, you can use techniques such as regularization, early stopping, or increasing the size of your training dataset.

TEST 2
Question: What is the difference between supervised and unsupervised learning?
----------------------------------------------------------------------
In machine learning, there are two types of algorithms: supervised learning and unsupervised learning. The main difference between them is that in supervised learning, we have labeled data (i.e., training examples with known labels) to train our mo

In [15]:
# Keep your current fine-tuned model safe
fine_tuned_model = model

# Load a separate base model
base_model_only = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    dtype=torch.float16,
    device_map="auto"
)

base_model_only.eval()

print("✅ Base model loaded!")

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

✅ Base model loaded!


In [16]:
def generate_base_response(prompt, max_new_tokens=150):
    messages = [
        {
            "role": "user",
            "content": prompt
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(base_model_only.device)

    with torch.no_grad():
        outputs = base_model_only.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

In [17]:
test_prompt = "Explain overfitting in simple words."

print("=" * 70)
print("BASE MODEL")
print("=" * 70)

print(generate_base_response(test_prompt))

print("\n" + "=" * 70)
print("FINE-TUNED MODEL")
print("=" * 70)

print(generate_response(test_prompt))

BASE MODEL
Overfitting is when a model learns the training data too well and starts to fit not just the underlying pattern but also random noise or outliers in the data. As a result, it performs poorly on new, unseen data because it has memorized the details of the training set rather than generalizing from those patterns. Essentially, the model becomes overly complex and sensitive to small fluctuations in the training data.

FINE-TUNED MODEL
Overfitting is when your machine learning model learns too much from the data you use to train it and doesn't generalize well to new data. It's like if you were trying to guess someone's age based only on their name - you might be able to make some good guesses for people with common names, but you wouldn't do very well at guessing ages of everyone else. Overfitting means that your model has learned all of the patterns in your training data so well that it starts to memorize them instead of actually understanding the underlying structure of the da

In [18]:
test_cases = [
    {
        "instruction": "Explain the concept of gradient descent to a beginner.",
        "context": ""
    },
    {
        "instruction": "Give three practical applications of machine learning.",
        "context": ""
    },
    {
        "instruction": "Summarize the following text in two sentences.",
        "context": "Machine learning allows computers to learn patterns from data. These patterns can then be used to make predictions or decisions on new data."
    },
    {
        "instruction": "Write a short professional email asking for an internship opportunity.",
        "context": ""
    },
    {
        "instruction": "Explain why validation data is important when training a machine learning model.",
        "context": ""
    }
]

def make_prompt(item):
    if item["context"]:
        return f"""### Instruction:
{item["instruction"]}

### Context:
{item["context"]}

### Response:"""
    else:
        return f"""### Instruction:
{item["instruction"]}

### Response:"""

In [19]:
for i, item in enumerate(test_cases, 1):

    prompt = make_prompt(item)

    print("\n" + "=" * 80)
    print(f"TEST {i}")
    print("=" * 80)
    print("Instruction:", item["instruction"])

    if item["context"]:
        print("Context:", item["context"])

    print("\nResponse:")
    print(generate_response(prompt, max_new_tokens=180))


TEST 1
Instruction: Explain the concept of gradient descent to a beginner.

Response:
Gradient descent is an optimization algorithm that can be used to find the minimum value of a function. It works by iteratively adjusting the parameters of the function in the direction of steepest decrease of the function's value. This process continues until the change in the function's value becomes small enough and stops further adjustments.

The idea behind gradient descent is simple but powerful. The function you want to optimize has many local minima, and one way to ensure that your algorithm finds the global minimum is to start at random points near each local minimum and then adjust the parameter values based on how far away they are from the local minimum. Over time, this will cause the algorithm to converge towards the global minimum.

In practice, gradient descent requires evaluating the function at multiple points to calculate its gradient (the vector pointing in the direction of the gre

In [20]:
resume_tests = [
    """Analyze this resume skill list and suggest suitable job roles:
C++, Python, React, Node.js, MongoDB, SQL, Machine Learning, Git.""",

    """Improve this resume bullet:
Worked on a web application using React and Node.js.""",

    """Identify skill gaps for a student who knows C++, DSA, React, Node.js and MongoDB but wants to become a Machine Learning Engineer.""",

    """Create 5 technical interview questions for a candidate who knows C++, DSA and Machine Learning.""",

    """Rewrite this resume bullet in a professional and impact-oriented way:
Made a website for managing student records."""
]

for i, prompt in enumerate(resume_tests, 1):
    print("\n" + "=" * 80)
    print(f"RESUME TEST {i}")
    print("=" * 80)
    print("PROMPT:", prompt)
    print("\nRESPONSE:")
    print(generate_response(prompt, max_new_tokens=220))


RESUME TEST 1
PROMPT: Analyze this resume skill list and suggest suitable job roles:
C++, Python, React, Node.js, MongoDB, SQL, Machine Learning, Git.

RESPONSE:
Based on the skills listed in the resume, you may be well-suited for positions that involve software development or data analysis. Here's how each language fits into different types of jobs:

- C++: This is often used in game development, systems programming, and large-scale applications.
- Python: A versatile language used in web development (React), machine learning, scientific computing, and more.
- React: Primarily used for building user interfaces in front-end development.
- Node.js: Used for server-side JavaScript in back-end development.
- MongoDB: An open-source document-oriented database management system.
- SQL: Generally used for managing relational databases.
- Machine Learning: Requires experience with algorithms and libraries such as TensorFlow, PyTorch, Scikit-Learn, etc.
- Git: Essential tool for version contr

In [21]:
import os

adapter_path = "/content/drive/MyDrive/qwen-dolly-lora"

print("adapter_config:", os.path.exists(
    adapter_path + "/adapter_config.json"
))

print("adapter_weights:", os.path.exists(
    adapter_path + "/adapter_model.safetensors"
))

adapter_config: True
adapter_weights: True


In [22]:
print(type(model))
print(model.peft_config)

<class 'peft.peft_model.PeftModelForCausalLM'>
{'default': LoraConfig(task_type='CAUSAL_LM', peft_type=<PeftType.LORA: 'LORA'>, auto_mapping=None, peft_version='0.20.0', base_model_name_or_path='Qwen/Qwen2.5-1.5B-Instruct', revision=None, inference_mode=True, r=16, target_modules={'q_proj', 'o_proj', 'v_proj', 'k_proj'}, exclude_modules=None, lora_alpha=32, lora_dropout=0.05, fan_in_fan_out=False, bias='none', use_rslora=False, modules_to_save=None, init_lora_weights=True, layers_to_transform=None, layers_pattern=None, rank_pattern={}, alpha_pattern={}, megatron_config=None, megatron_core='megatron.core', trainable_token_indices=None, loftq_config={}, eva_config=None, corda_config=None, lora_ga_config=None, use_dora=False, velora_config=None, alora_invocation_tokens=None, use_qalora=False, qalora_group_size=16, monteclora_config=None, layer_replication=None, runtime_config=LoraRuntimeConfig(ephemeral_gpu_offload=False), lora_bias=False, target_parameters=None, use_bdlora=None, arrow_co

In [23]:
print("Base model parameters:", sum(
    p.numel() for p in base_model.parameters()
))

print("LoRA trainable parameters:")

model.print_trainable_parameters()

Base model parameters: 1548072448
LoRA trainable parameters:
trainable params: 0 || all params: 1,548,072,448 || trainable%: 0.0000


In [24]:
prompt = """### Instruction:
Explain the difference between supervised and unsupervised learning in exactly 3 bullet points.

### Response:"""

print("=" * 80)
print("BASE MODEL")
print("=" * 80)

base_answer = generate_base_response(
    prompt,
    max_new_tokens=150
)

print(base_answer)


print("\n" + "=" * 80)
print("FINE-TUNED MODEL")
print("=" * 80)

fine_answer = generate_response(
    prompt,
    max_new_tokens=150
)

print(fine_answer)

BASE MODEL
1. **Supervised Learning** involves training an algorithm on labeled data, where each example has both input features and corresponding output labels. The goal is to learn a function that maps inputs to outputs accurately based on these labeled examples. Common tasks include classification (predicting categorical outcomes) or regression (predicting continuous values).

2. **Unsupervised Learning**, conversely, deals with unlabeled data—data without any predefined output variables. Its primary objective is to find patterns or structure within the data itself. Examples of unsupervised learning algorithms include clustering (grouping similar instances together), dimensionality reduction (reducing the number of random variables under consideration), and anomaly detection (identifying unusual data points).

3. **Difference**: In supervised

FINE-TUNED MODEL
Supervised learning is when you have labeled training data, where each example has an input value and a corresponding output

In [25]:
import os
from safetensors.torch import load_file

adapter_file = "/content/drive/MyDrive/qwen-dolly-lora/adapter_model.safetensors"

print("Adapter exists:", os.path.exists(adapter_file))

adapter_state = load_file(adapter_file)

print("Number of LoRA tensors:", len(adapter_state))
print("\nFirst 10 tensors:")

for name, tensor in list(adapter_state.items())[:10]:
    print(name, tensor.shape, tensor.dtype)

Adapter exists: True
Number of LoRA tensors: 224

First 10 tensors:
base_model.model.model.layers.0.self_attn.k_proj.lora_A.weight torch.Size([16, 1536]) torch.float32
base_model.model.model.layers.0.self_attn.k_proj.lora_B.weight torch.Size([256, 16]) torch.float32
base_model.model.model.layers.0.self_attn.o_proj.lora_A.weight torch.Size([16, 1536]) torch.float32
base_model.model.model.layers.0.self_attn.o_proj.lora_B.weight torch.Size([1536, 16]) torch.float32
base_model.model.model.layers.0.self_attn.q_proj.lora_A.weight torch.Size([16, 1536]) torch.float32
base_model.model.model.layers.0.self_attn.q_proj.lora_B.weight torch.Size([1536, 16]) torch.float32
base_model.model.model.layers.0.self_attn.v_proj.lora_A.weight torch.Size([16, 1536]) torch.float32
base_model.model.model.layers.0.self_attn.v_proj.lora_B.weight torch.Size([256, 16]) torch.float32
base_model.model.model.layers.1.self_attn.k_proj.lora_A.weight torch.Size([16, 1536]) torch.float32
base_model.model.model.layers.1.se

In [26]:
import os

path = "/content/drive/MyDrive/qwen-dolly-lora"

print(os.listdir(path))

['checkpoint-1600', 'checkpoint-1689', 'tokenizer.json', 'adapter_model.safetensors', 'README.md', 'training_args.bin', 'adapter_config.json', 'tokenizer_config.json', 'chat_template.jinja']


In [27]:
import os

path = "/content/drive/MyDrive/qwen-dolly-lora/adapter_model.safetensors"

print("Adapter size:",
      round(os.path.getsize(path) / (1024**2), 2),
      "MB")

Adapter size: 16.65 MB


In [28]:
evaluation_questions = [
    {
        "id": 1,
        "question": "Explain overfitting in simple words with a real-life example."
    },
    {
        "id": 2,
        "question": "What is the difference between supervised and unsupervised learning?"
    },
    {
        "id": 3,
        "question": "Explain gradient descent to a beginner."
    },
    {
        "id": 4,
        "question": "What is the purpose of a validation dataset?"
    },
    {
        "id": 5,
        "question": "Why is feature scaling important in machine learning?"
    },
    {
        "id": 6,
        "question": "Explain precision and recall with a simple example."
    },
    {
        "id": 7,
        "question": "What is the difference between classification and regression?"
    },
    {
        "id": 8,
        "question": "Explain neural networks in simple terms."
    },
    {
        "id": 9,
        "question": "What is an epoch in deep learning?"
    },
    {
        "id": 10,
        "question": "Why do we use a learning rate in gradient descent?"
    },
    {
        "id": 11,
        "question": "Write a professional email asking for an internship opportunity."
    },
    {
        "id": 12,
        "question": "Summarize the concept of machine learning in exactly two sentences."
    },
    {
        "id": 13,
        "question": "Give three practical applications of deep learning."
    },
    {
        "id": 14,
        "question": "Explain the difference between training loss and validation loss."
    },
    {
        "id": 15,
        "question": "What happens when a model has very high variance?"
    },
    {
        "id": 16,
        "question": "Explain transfer learning and why it is useful."
    },
    {
        "id": 17,
        "question": "What is fine-tuning in the context of large language models?"
    },
    {
        "id": 18,
        "question": "Explain LoRA in simple words."
    },
    {
        "id": 19,
        "question": "Why can a model perform well on training data but poorly on new data?"
    },
    {
        "id": 20,
        "question": "Explain the difference between a test set and a validation set."
    }
]

print("Total evaluation questions:", len(evaluation_questions))

Total evaluation questions: 20


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [1]:
import torch
from transformers import AutoModelForCausalLM

# -----------------------------
# Base Model
# -----------------------------
print("Loading base model...")

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    dtype=torch.float16,
    device_map="auto"
)

base_model.eval()

print("✅ Base model loaded")


# -----------------------------
# Generation function
# -----------------------------
def generate_model_answer(model, question, max_new_tokens=150):
    messages = [
        {"role": "user", "content": question}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )


# -----------------------------
# Generate answers
# -----------------------------
evaluation_results = []

for item in evaluation_questions:

    question = item["question"]

    print(f"Processing Q{item['id']}...")

    base_answer = generate_model_answer(
        base_model,
        question
    )

    finetuned_answer = generate_model_answer(
        model,
        question
    )

    evaluation_results.append({
        "id": item["id"],
        "question": question,
        "base_answer": base_answer,
        "finetuned_answer": finetuned_answer
    })

print("\n✅ Evaluation answers generated!")
print("Total questions:", len(evaluation_results))

Loading base model...


NameError: name 'base_model_name' is not defined

In [2]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("❌ GPU not available")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [3]:
base_model_name = "Qwen/Qwen2.5-1.5B-Instruct"

adapter_path = "/content/drive/MyDrive/qwen-dolly-lora"

print("Base model:", base_model_name)
print("Adapter path:", adapter_path)

Base model: Qwen/Qwen2.5-1.5B-Instruct
Adapter path: /content/drive/MyDrive/qwen-dolly-lora


In [4]:
from transformers import AutoModelForCausalLM
import torch

print("Loading base model...")

base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    dtype=torch.float16,
    device_map="auto"
)

base_model.eval()

print("✅ Base model loaded successfully!")
print("Device:", base_model.device)

Loading base model...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

✅ Base model loaded successfully!
Device: cuda:0


In [5]:
from peft import PeftModel

print("Loading LoRA adapter...")

finetuned_model = PeftModel.from_pretrained(
    base_model,
    adapter_path
)

finetuned_model.eval()

print("✅ Fine-tuned model loaded successfully!")

Loading LoRA adapter...


ValueError: Can't find 'adapter_config.json' at '/content/drive/MyDrive/qwen-dolly-lora'

In [6]:
import os

print("Adapter folder exists:", os.path.exists(adapter_path))
print("\nFiles inside adapter folder:")

for file in os.listdir(adapter_path):
    print(file)

Adapter folder exists: False

Files inside adapter folder:


FileNotFoundError: [Errno 2] No such file or directory: '/content/drive/MyDrive/qwen-dolly-lora'

In [7]:
from google.colab import drive

drive.mount('/content/drive')

Mounted at /content/drive


In [8]:
import os

adapter_path = "/content/drive/MyDrive/qwen-dolly-lora"

print("Adapter folder exists:", os.path.exists(adapter_path))

if os.path.exists(adapter_path):
    print("\nFiles:")
    for file in os.listdir(adapter_path):
        print(file)

Adapter folder exists: True

Files:
checkpoint-1600
checkpoint-1689
tokenizer.json
adapter_model.safetensors
README.md
training_args.bin
adapter_config.json
tokenizer_config.json
chat_template.jinja


In [9]:
from peft import PeftModel

print("Loading LoRA adapter...")

finetuned_model = PeftModel.from_pretrained(
    base_model,
    adapter_path
)

finetuned_model.eval()

print("✅ Fine-tuned model loaded successfully!")

Loading LoRA adapter...


ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

In [10]:
adapter_path = "/content/drive/MyDrive/qwen-dolly-lora"

In [11]:
from peft import PeftModel

print("Loading LoRA adapter...")

finetuned_model = PeftModel.from_pretrained(
    base_model,
    adapter_path
)

finetuned_model.eval()

print("✅ Fine-tuned model loaded successfully!")

Loading LoRA adapter...


ImportError: Found an incompatible version of torchao. Found version 0.10.0, but only versions above 0.16.0 are supported

In [12]:
!pip uninstall -y torchao

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0


In [13]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("❌ GPU not available")

PyTorch: 2.11.0+cu128
CUDA available: True
GPU: Tesla T4


In [14]:
base_model_name = "Qwen/Qwen2.5-1.5B-Instruct"
adapter_path = "/content/drive/MyDrive/qwen-dolly-lora"

print("Base model:", base_model_name)
print("Adapter path:", adapter_path)

Base model: Qwen/Qwen2.5-1.5B-Instruct
Adapter path: /content/drive/MyDrive/qwen-dolly-lora


In [15]:
from google.colab import drive
drive.mount("/content/drive")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [16]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(base_model_name)

print("✅ Tokenizer loaded")

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

✅ Tokenizer loaded


In [17]:
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_name,
    dtype=torch.float16,
    device_map="auto"
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [18]:
from peft import PeftModel

finetuned_model = PeftModel.from_pretrained(
    base_model,
    adapter_path
)

finetuned_model.eval()

print("✅ Fine-tuned model loaded")

✅ Fine-tuned model loaded


In [19]:
import torch

def generate_model_answer(model, question, max_new_tokens=150):
    messages = [
        {"role": "user", "content": question}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

print("✅ Generation function ready")

✅ Generation function ready


In [20]:
evaluation_questions = [
    {"id": 1, "question": "Explain overfitting in simple words with a real-life example."},
    {"id": 2, "question": "What is the difference between supervised and unsupervised learning?"},
    {"id": 3, "question": "Explain gradient descent to a beginner."},
    {"id": 4, "question": "What is the purpose of a validation dataset?"},
    {"id": 5, "question": "Why is feature scaling important in machine learning?"},
    {"id": 6, "question": "Explain precision and recall with a simple example."},
    {"id": 7, "question": "What is the difference between classification and regression?"},
    {"id": 8, "question": "Explain neural networks in simple terms."},
    {"id": 9, "question": "What is an epoch in deep learning?"},
    {"id": 10, "question": "Why do we use a learning rate in gradient descent?"},
    {"id": 11, "question": "Write a professional email asking for an internship opportunity."},
    {"id": 12, "question": "Summarize the concept of machine learning in exactly two sentences."},
    {"id": 13, "question": "Give three practical applications of deep learning."},
    {"id": 14, "question": "Explain the difference between training loss and validation loss."},
    {"id": 15, "question": "What happens when a model has very high variance?"},
    {"id": 16, "question": "Explain transfer learning and why it is useful."},
    {"id": 17, "question": "What is fine-tuning in the context of large language models?"},
    {"id": 18, "question": "Explain LoRA in simple words."},
    {"id": 19, "question": "Why can a model perform well on training data but poorly on new data?"},
    {"id": 20, "question": "Explain the difference between a test set and a validation set."}
]

print("Total evaluation questions:", len(evaluation_questions))

Total evaluation questions: 20


In [21]:
evaluation_results = []

for item in evaluation_questions:

    question = item["question"]

    print(f"Processing Q{item['id']}/20...")

    # Base model
    base_answer = generate_model_answer(
        base_model,
        question
    )

    # Fine-tuned model
    finetuned_answer = generate_model_answer(
        finetuned_model,
        question
    )

    evaluation_results.append({
        "id": item["id"],
        "question": question,
        "base_answer": base_answer,
        "finetuned_answer": finetuned_answer
    })

print("\n✅ Evaluation completed!")
print("Total questions evaluated:", len(evaluation_results))

Processing Q1/20...
Processing Q2/20...
Processing Q3/20...
Processing Q4/20...
Processing Q5/20...
Processing Q6/20...
Processing Q7/20...
Processing Q8/20...
Processing Q9/20...
Processing Q10/20...
Processing Q11/20...
Processing Q12/20...
Processing Q13/20...
Processing Q14/20...
Processing Q15/20...
Processing Q16/20...
Processing Q17/20...
Processing Q18/20...
Processing Q19/20...
Processing Q20/20...

✅ Evaluation completed!
Total questions evaluated: 20


In [22]:
for result in evaluation_results[:3]:
    print("=" * 80)
    print(f"Q{result['id']}: {result['question']}")

    print("\n--- BASE MODEL ---")
    print(result["base_answer"])

    print("\n--- FINE-TUNED MODEL ---")
    print(result["finetuned_answer"])

    print()

Q1: Explain overfitting in simple words with a real-life example.

--- BASE MODEL ---
Overfitting is when an algorithm fits the training data too closely and performs well on that particular dataset but poorly on new, unseen data. It's like if you were drawing pictures of your friends only using photos of them as reference; while you can draw their exact likeness in your drawings, it would be difficult for anyone else to recognize those pictures unless they have access to your original photos. Similarly, an overfitted model will perform exceptionally well on its own training data but might not generalize well to other datasets or scenarios.

A real-life example could be predicting stock prices. If you use historical stock price data to build a model that predicts future prices very accurately based solely on past trends, it may work great when tested against similar data from previous years

--- FINE-TUNED MODEL ---
Overfitting is when your model learns too much from the data you have 

In [23]:
def generate_model_answer(model, question, max_new_tokens=250):
    messages = [
        {"role": "user", "content": question}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=0.7,
            top_p=0.9,
            do_sample=True,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

print("✅ Updated generation function")

✅ Updated generation function


In [24]:
evaluation_results = []

for item in evaluation_questions:

    question = item["question"]

    print(f"Processing Q{item['id']}/20...")

    base_answer = generate_model_answer(
        base_model,
        question,
        max_new_tokens=250
    )

    finetuned_answer = generate_model_answer(
        finetuned_model,
        question,
        max_new_tokens=250
    )

    evaluation_results.append({
        "id": item["id"],
        "question": question,
        "base_answer": base_answer,
        "finetuned_answer": finetuned_answer
    })

print("\n✅ Evaluation completed!")
print("Total:", len(evaluation_results))

Processing Q1/20...
Processing Q2/20...
Processing Q3/20...
Processing Q4/20...
Processing Q5/20...
Processing Q6/20...
Processing Q7/20...
Processing Q8/20...
Processing Q9/20...
Processing Q10/20...
Processing Q11/20...
Processing Q12/20...
Processing Q13/20...
Processing Q14/20...
Processing Q15/20...
Processing Q16/20...
Processing Q17/20...
Processing Q18/20...
Processing Q19/20...
Processing Q20/20...

✅ Evaluation completed!
Total: 20


In [25]:
import pandas as pd

results_df = pd.DataFrame(evaluation_results)

display(
    results_df[
        ["id", "question", "base_answer", "finetuned_answer"]
    ]
)

,id,question,base_answer,finetuned_answer
0,1,Explain overfitting in simple words with a rea...,Overfitting is a common problem that occurs wh...,Overfitting is when a machine learning model l...
1,2,What is the difference between supervised and ...,Supervised learning: The machine learning mode...,Supervised learning and unsupervised learning ...
2,3,Explain gradient descent to a beginner.,Gradient descent is an optimization algorithm ...,Gradient Descent is an optimization algorithm ...
3,4,What is the purpose of a validation dataset?,A validation dataset is used to evaluate and c...,A validation dataset is used to evaluate the p...
4,5,Why is feature scaling important in machine le...,Feature scaling is an essential step before ap...,Feature scaling is an important preprocessing ...
5,6,Explain precision and recall with a simple exa...,Precision and Recall are two common metrics us...,Precision is the proportion of true positives ...
6,7,What is the difference between classification ...,Classification and regression are two types of...,Classification and regression are both methods...
7,8,Explain neural networks in simple terms.,Neural networks can be thought of as complex m...,Neural networks are machine learning models in...
8,9,What is an epoch in deep learning?,An epoch (also known as one pass through the d...,An epoch is the term used to describe one comp...
9,10,Why do we use a learning rate in gradient desc...,"In gradient descent algorithms, the goal is to...","In machine learning and data analysis, the lea..."


In [26]:
scored_results = []

for result in evaluation_results:
    scored_results.append({
        "id": result["id"],
        "question": result["question"],

        # Base model
        "base_correctness": None,
        "base_instruction": None,
        "base_relevance": None,
        "base_clarity": None,

        # Fine-tuned model
        "ft_correctness": None,
        "ft_instruction": None,
        "ft_relevance": None,
        "ft_clarity": None,
    })

print("✅ Scoring sheet created")
print("Questions:", len(scored_results))

✅ Scoring sheet created
Questions: 20


In [27]:
import json
import re
import random
import torch

def score_answers(question, answer_a, answer_b):
    prompt = f"""
You are a strict evaluator comparing two answers to the same question.

QUESTION:
{question}

ANSWER A:
{answer_a}

ANSWER B:
{answer_b}

Score EACH answer independently using these four criteria:

1. Correctness: 0-5
2. Instruction Following: 0-5
3. Relevance: 0-5
4. Clarity: 0-5

Scoring guide:
0 = completely poor
1 = very weak
2 = below average
3 = acceptable
4 = good
5 = excellent

Do not decide which answer is better overall.
Evaluate each answer independently.

Return ONLY valid JSON in exactly this format:

{{
  "A": {{
    "correctness": 0,
    "instruction": 0,
    "relevance": 0,
    "clarity": 0
  }},
  "B": {{
    "correctness": 0,
    "instruction": 0,
    "relevance": 0,
    "clarity": 0
  }}
}}
"""

    messages = [
        {"role": "user", "content": prompt}
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(base_model.device)

    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            temperature=None,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    # Extract JSON
    match = re.search(r'\{.*\}', response, re.DOTALL)

    if not match:
        return None

    try:
        return json.loads(match.group())
    except:
        return None


print("✅ Evaluator ready")

✅ Evaluator ready


In [28]:
q1 = evaluation_results[0]

score_q1 = score_answers(
    q1["question"],
    q1["base_answer"],
    q1["finetuned_answer"]
)

print(json.dumps(score_q1, indent=2))

{
  "A": {
    "correctness": 0,
    "instruction": 0,
    "relevance": 0,
    "clarity": 0
  },
  "B": {
    "correctness": 0,
    "instruction": 0,
    "relevance": 0,
    "clarity": 0
  }
}


In [29]:
def debug_score_answers(question, answer_a, answer_b):

    prompt = f"""
You are evaluating two answers.

QUESTION:
{question}

ANSWER A:
{answer_a}q1 = evaluation_results[0]

debug_score_answers(
    q1["question"],
    q1["base_answer"],
    q1["finetuned_answer"]
)

ANSWER B:
{answer_b}

For EACH answer, give a score from 0 to 5 for:

- correctness
- instruction following
- relevance
- clarity

5 = excellent
4 = good
3 = acceptable
2 = weak
1 = very weak
0 = completely incorrect or irrelevant

Return ONLY JSON:

{{
  "A": {{
    "correctness": 0,
    "instruction": 0,
    "relevance": 0,
    "clarity": 0
  }},
  "B": {{
    "correctness": 0,
    "instruction": 0,
    "relevance": 0,
    "clarity": 0
  }}
}}
"""

    messages = [{"role": "user", "content": prompt}]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(base_model.device)

    with torch.no_grad():
        outputs = base_model.generate(
            **inputs,
            max_new_tokens=200,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )

    generated_tokens = outputs[0][inputs["input_ids"].shape[1]:]

    response = tokenizer.decode(
        generated_tokens,
        skip_special_tokens=True
    )

    print("========== RAW EVALUATOR RESPONSE ==========")
    print(response)

In [30]:
q1 = evaluation_results[0]

debug_score_answers(
    q1["question"],
    q1["base_answer"],
    q1["finetuned_answer"]
)

========== RAW EVALUATOR RESPONSE ==========
{
  "A": {
    "correctness": 0,
    "instruction": 0,
    "relevance": 0,
    "clarity": 0
  },
  "B": {
    "correctness": 0,
    "instruction": 0,
    "relevance": 0,
    "clarity": 0
  }
}


In [31]:
import pandas as pd

manual_scores = pd.DataFrame({
    "id": range(1, 21),

    "base_correctness": [None] * 20,
    "base_instruction": [None] * 20,
    "base_relevance": [None] * 20,
    "base_clarity": [None] * 20,

    "ft_correctness": [None] * 20,
    "ft_instruction": [None] * 20,
    "ft_relevance": [None] * 20,
    "ft_clarity": [None] * 20,
})

manual_scores["base_total"] = None
manual_scores["ft_total"] = None

display(manual_scores)

,id,base_correctness,base_instruction,base_relevance,base_clarity,ft_correctness,ft_instruction,ft_relevance,ft_clarity,base_total,ft_total
0,1,None,None,None,None,None,None,None,None,None,None
1,2,None,None,None,None,None,None,None,None,None,None
2,3,None,None,None,None,None,None,None,None,None,None
3,4,None,None,None,None,None,None,None,None,None,None
4,5,None,None,None,None,None,None,None,None,None,None
5,6,None,None,None,None,None,None,None,None,None,None
6,7,None,None,None,None,None,None,None,None,None,None
7,8,None,None,None,None,None,None,None,None,None,None
8,9,None,None,None,None,None,None,None,None,None,None
9,10,None,None,None,None,None,None,None,None,None,None


In [32]:
for result in evaluation_results[:5]:
    print("=" * 100)
    print(f"Q{result['id']}: {result['question']}")

    print("\n--- BASE MODEL ---")
    print(result["base_answer"])

    print("\n--- FINE-TUNED MODEL ---")
    print(result["finetuned_answer"])
    print()

Q1: Explain overfitting in simple words with a real-life example.

--- BASE MODEL ---
Overfitting is a common problem that occurs when a machine learning model learns too much from the training data and starts to fit to noise rather than the underlying pattern. This can happen if the model has been trained for too long or on a very large dataset, which means it memorizes every detail of the training data instead of generalizing to new examples.

A real-life example of overfitting could be someone who is trying to learn how to play chess. If they practice playing against themselves using the same moves over and over again without changing their strategy or tactics, they might end up getting better at memorizing the patterns of their own moves but not necessarily improving their overall gameplay. In other words, they may have learned how to beat themselves, but they haven't actually improved their ability to play well against more experienced players.

In machine learning terms, this wou

In [33]:
# Q1
manual_scores.loc[0, [
    "base_correctness", "base_instruction",
    "base_relevance", "base_clarity",
    "ft_correctness", "ft_instruction",
    "ft_relevance", "ft_clarity"
]] = [5, 5, 5, 4, 5, 5, 5, 4]

# Q2
manual_scores.loc[1, [
    "base_correctness", "base_instruction",
    "base_relevance", "base_clarity",
    "ft_correctness", "ft_instruction",
    "ft_relevance", "ft_clarity"
]] = [5, 5, 5, 5, 5, 5, 5, 5]

print("✅ Q1 and Q2 scored")

✅ Q1 and Q2 scored


In [34]:
for result in evaluation_results[2:5]:
    print("=" * 100)
    print(f"Q{result['id']}: {result['question']}")
    print("\n--- BASE MODEL ---")
    print(result["base_answer"])
    print("\n--- FINE-TUNED MODEL ---")
    print(result["finetuned_answer"])
    print()

Q3: Explain gradient descent to a beginner.

--- BASE MODEL ---
Gradient descent is an optimization algorithm used in machine learning and deep learning for minimizing the error between predictions made by a model and actual data points. It's commonly used when you have a large dataset with many parameters that need to be adjusted to find the best fit.
The basic idea behind gradient descent is to start with some initial guess of the parameter values, then iteratively adjust those values based on how well they minimize the error. The direction in which to adjust the parameters depends on the sign of the gradient (which measures the rate of change) of the loss function at each point. By following this path down the hill towards the lowest point, you can gradually converge on the optimal set of parameter values that minimizes the overall loss.
There are several variations of gradient descent, including batch gradient descent, stochastic gradient descent, and mini-batch gradient descent, d

In [35]:
import pandas as pd

evaluation_df = pd.DataFrame(evaluation_results)

evaluation_df = evaluation_df[
    ["id", "question", "base_answer", "finetuned_answer"]
]

evaluation_df.to_csv(
    "/content/evaluation_results.csv",
    index=False
)

print("✅ Evaluation file created!")
print("Rows:", len(evaluation_df))
print("File: /content/evaluation_results.csv")

✅ Evaluation file created!
Rows: 20
File: /content/evaluation_results.csv
